# 01 Threshold + Morphology：MVTec AD Tile 瑕疵檢測第一版實驗

本 Notebook 是 `industrial_tile_defect_detection` 專案的第一版實驗，目標是用最直覺、可解釋的傳統影像處理方法，嘗試從 MVTec AD 的 `tile` 圖像中定位表面瑕疵區域。

這一版的核心想法很單純：

> 正常 tile 圖像中有大量深色小斑點，這些斑點是磁磚本身的紋理，不應該被視為瑕疵。  
> 真正的瑕疵通常會比正常斑點更大、更連續、更集中，或在形狀與區域結構上明顯不同。  
> 因此，本實驗先使用二值化取得候選區域，再透過 morphology 與 connected components 移除小型正常紋理，保留較可能屬於瑕疵的區域。

這不是最終方法，而是第一個可解釋版本。  
本 Notebook 會完整呈現這個想法在不同瑕疵類型上的效果，並誠實分析它適合哪些情況、在哪些情況會失敗，以及下一版應該如何改善。

## 0. 專案資料結構與使用方式

本 Notebook 預期放在：

```text
industrial_tile_defect_detection/
├─ notebooks/
│  └─ 01_threshold_morphology.ipynb
├─ data/
│  ├─ good/
│  ├─ defect/
│  │  ├─ crack/
│  │  ├─ glue_strip/
│  │  ├─ gray_stroke/
│  │  ├─ oil/
│  │  └─ rough/
│  └─ ground_truth/
│     ├─ crack/
│     ├─ glue_strip/
│     ├─ gray_stroke/
│     ├─ oil/
│     └─ rough/
└─ outputs/
   ├─ figures/
   ├─ masks/
   └─ overlays/
```

### Ground truth mask 命名規則

本 Notebook 會自動嘗試用下列規則尋找 ground truth mask：

```text
data/ground_truth/{defect_type}/{image_stem}_mask.png
data/ground_truth/{defect_type}/{image_stem}.png
data/ground_truth/{defect_type}/{image_stem}_mask.bmp
data/ground_truth/{defect_type}/{image_stem}.bmp
```

例如：

```text
data/defect/crack/000.png
data/ground_truth/crack/000_mask.png
```

若找不到 ground truth，Notebook 仍可執行，只是不會計算 IoU、Dice、Precision、Recall 等 mask-level 指標。

## 1. Dataset Citation

本專案使用 MVTec AD dataset 的 `tile` 類別。  
若此專案上傳至 GitHub、作品集、報告或任何公開文件，應引用原資料集論文。

```bibtex
@inproceedings{bergmann2019mvtec,
  title={MVTec AD -- A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection},
  author={Bergmann, Paul and Fauser, Michael and Sattlegger, David and Steger, Carsten},
  booktitle={Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition},
  year={2019}
}
```

本 Notebook 僅使用少量樣本進行方法展示與工程分析，不重新發布完整資料集。

## 2. Environment Setup

這一版只使用常見的影像處理與資料分析套件：

- `opencv-python`：影像讀取、二值化、morphology、contour、connected components
- `numpy`：陣列運算
- `matplotlib`：圖像視覺化
- `pandas`：結果表格整理
- `scikit-image`：部分輔助量測與圖像處理

若尚未安裝，可在專案根目錄執行：

```bash
pip install opencv-python numpy matplotlib pandas scikit-image
```

In [ ]:
from pathlib import Path
import os
import math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import measure

plt.rcParams["figure.dpi"] = 120

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"
GOOD_DIR = DATA_DIR / "good"
DEFECT_DIR = DATA_DIR / "defect"
GT_DIR = DATA_DIR / "ground_truth"

VERSION_NAME = "01_threshold_morphology"
OUTPUT_ROOT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR = OUTPUT_ROOT_DIR / VERSION_NAME
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MASK_DIR = OUTPUT_DIR / "masks"
OVERLAY_DIR = OUTPUT_DIR / "overlays"

for d in [FIGURE_DIR, TABLE_DIR, MASK_DIR, OVERLAY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEFECT_TYPES = ["crack", "glue_strip", "gray_stroke", "oil", "rough"]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

print("PROJECT_DIR:", PROJECT_DIR.resolve())
print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 3. Utility Functions

這一節先建立本 Notebook 會用到的工具函式。  
因為本專案目前不使用 `src/`，所有實驗流程都集中在這個 Notebook 中。

這些函式分成幾類：

1. 資料讀取與路徑管理  
2. 圖像顯示與儲存  
3. 二值化與 morphology  
4. connected components 與 bounding box  
5. ground truth 比對與評估指標

In [ ]:
def list_images(folder: Path):
    """列出資料夾中的影像檔案。"""
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])


def build_dataset_index():
    """建立 good 與 defect 影像清單。"""
    records = []

    for p in list_images(GOOD_DIR):
        records.append({
            "image_path": p,
            "label": "good",
            "defect_type": "good",
            "image_name": p.name
        })

    for defect_type in DEFECT_TYPES:
        folder = DEFECT_DIR / defect_type
        for p in list_images(folder):
            records.append({
                "image_path": p,
                "label": "defect",
                "defect_type": defect_type,
                "image_name": p.name
            })

    return pd.DataFrame(records)


def read_rgb(path: Path):
    """使用 OpenCV 讀取影像，並轉成 RGB。"""
    img_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f"無法讀取影像：{path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def to_gray(rgb):
    """RGB 轉灰階。"""
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)


def find_ground_truth(image_path: Path, defect_type: str):
    """
    根據影像名稱尋找 ground truth mask。
    good image 沒有 defect mask，直接回傳 None。
    """
    if defect_type == "good":
        return None

    gt_folder = GT_DIR / defect_type
    if not gt_folder.exists():
        return None

    stem = image_path.stem
    candidates = [
        gt_folder / f"{stem}_mask.png",
        gt_folder / f"{stem}.png",
        gt_folder / f"{stem}_mask.bmp",
        gt_folder / f"{stem}.bmp",
        gt_folder / f"{stem}_mask.jpg",
        gt_folder / f"{stem}.jpg",
    ]

    for c in candidates:
        if c.exists():
            return c

    fallback = sorted(list(gt_folder.glob(f"{stem}*")))
    fallback = [p for p in fallback if p.suffix.lower() in IMAGE_EXTENSIONS]
    return fallback[0] if fallback else None


def read_gt_mask(gt_path: Path, target_shape):
    """讀取 ground truth mask，轉為 0/1 binary mask。"""
    if gt_path is None:
        return None

    gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
    if gt is None:
        return None

    if gt.shape != target_shape:
        gt = cv2.resize(gt, (target_shape[1], target_shape[0]), interpolation=cv2.INTER_NEAREST)

    return (gt > 0).astype(np.uint8)


def show_image_grid(items, cols=3, figsize=(12, 8), save_path=None):
    """
    顯示影像 grid。
    items: list of (title, image, cmap)
    """
    if len(items) == 0:
        print("沒有影像可顯示。")
        return

    rows = math.ceil(len(items) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (title, img, cmap) in zip(axes, items):
        if img.ndim == 2:
            ax.imshow(img, cmap=cmap or "gray")
        else:
            ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()


def overlay_mask(rgb, mask, color=(255, 0, 0), alpha=0.45):
    """將 binary mask 疊加到原圖上。"""
    overlay = rgb.copy()
    color_layer = np.zeros_like(rgb)
    color_layer[:, :] = color
    mask_bool = mask.astype(bool)
    blended = cv2.addWeighted(rgb, 1 - alpha, color_layer, alpha, 0)
    overlay[mask_bool] = blended[mask_bool]
    return overlay


def draw_boxes(rgb, components, color=(255, 0, 0), thickness=3):
    """根據 connected components 結果畫 bounding boxes。"""
    out = rgb.copy()
    for comp in components:
        x, y, w, h = comp["bbox"]
        cv2.rectangle(out, (x, y), (x + w, y + h), color, thickness)
    return out

In [ ]:
def manual_dark_threshold(gray, threshold_value=80):
    """
    手動暗區二值化。
    灰階值越低代表越暗，因此 gray < threshold_value 會被視為候選區域。
    """
    mask = (gray < threshold_value).astype(np.uint8)
    return mask


def otsu_dark_threshold(gray):
    """
    Otsu 二值化。
    因為我們想抓暗區，所以使用 THRESH_BINARY_INV。
    """
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return (mask > 0).astype(np.uint8)


def adaptive_dark_threshold(gray, block_size=51, c=5):
    """
    Adaptive threshold。
    用於局部亮度變化較明顯的情況。
    """
    if block_size % 2 == 0:
        block_size += 1
    mask = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        block_size,
        c
    )
    return (mask > 0).astype(np.uint8)


def apply_morphology(mask, open_kernel=5, close_kernel=15, open_iter=1, close_iter=1):
    """
    第一版 morphology 流程：
    1. opening: 移除小型孤立紋理點
    2. closing: 連接較破碎的瑕疵候選區域
    """
    mask_u8 = (mask > 0).astype(np.uint8) * 255

    if open_kernel > 0:
        k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel, open_kernel))
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, k_open, iterations=open_iter)

    if close_kernel > 0:
        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, k_close, iterations=close_iter)

    return (mask_u8 > 0).astype(np.uint8)


def extract_components(mask, min_area=500, max_area_ratio=0.8):
    """
    從 binary mask 中取得 connected components。
    使用 min_area 移除小區域，避免正常小黑點被視為 defect。
    """
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        (mask > 0).astype(np.uint8), connectivity=8
    )

    h, w = mask.shape
    image_area = h * w
    components = []

    for label_id in range(1, num_labels):
        x, y, bw, bh, area = stats[label_id]
        if area < min_area:
            continue
        if area > image_area * max_area_ratio:
            continue

        aspect_ratio = max(bw / max(bh, 1), bh / max(bw, 1))
        extent = area / max(bw * bh, 1)

        components.append({
            "label_id": label_id,
            "bbox": (int(x), int(y), int(bw), int(bh)),
            "area": int(area),
            "centroid": tuple(centroids[label_id]),
            "aspect_ratio": float(aspect_ratio),
            "extent": float(extent)
        })

    return components


def compute_mask_metrics(pred_mask, gt_mask):
    """計算 prediction mask 與 ground truth mask 的 pixel-level 指標。"""
    if gt_mask is None:
        return None

    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    tn = np.logical_and(~pred, ~gt).sum()

    iou = tp / (tp + fp + fn + 1e-8)
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    fpr = fp / (fp + tn + 1e-8)

    return {
        "iou": float(iou),
        "dice": float(dice),
        "precision": float(precision),
        "recall": float(recall),
        "false_positive_rate": float(fpr),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn)
    }


def run_threshold_morphology_pipeline(
    rgb,
    threshold_method="manual",
    threshold_value=80,
    adaptive_block_size=51,
    adaptive_c=5,
    open_kernel=5,
    close_kernel=15,
    min_area=500
):
    """
    完整第一版流程：
    RGB -> grayscale -> threshold -> morphology -> connected components -> final mask
    """
    gray = to_gray(rgb)

    if threshold_method == "manual":
        raw_mask = manual_dark_threshold(gray, threshold_value=threshold_value)
    elif threshold_method == "otsu":
        raw_mask = otsu_dark_threshold(gray)
    elif threshold_method == "adaptive":
        raw_mask = adaptive_dark_threshold(gray, block_size=adaptive_block_size, c=adaptive_c)
    else:
        raise ValueError(f"Unknown threshold_method: {threshold_method}")

    morph_mask = apply_morphology(
        raw_mask,
        open_kernel=open_kernel,
        close_kernel=close_kernel
    )

    components = extract_components(morph_mask, min_area=min_area)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        (morph_mask > 0).astype(np.uint8), connectivity=8
    )

    final_mask = np.zeros_like(morph_mask)
    kept_label_ids = [c["label_id"] for c in components]
    for label_id in kept_label_ids:
        final_mask[labels == label_id] = 1

    overlay = overlay_mask(rgb, final_mask, color=(255, 0, 0), alpha=0.45)
    boxed = draw_boxes(overlay, components)

    return {
        "gray": gray,
        "raw_mask": raw_mask,
        "morph_mask": morph_mask,
        "components": components,
        "final_mask": final_mask,
        "overlay": overlay,
        "boxed": boxed
    }

## 4. Dataset Loading and Structure Check

先檢查資料是否放置正確，並確認每一類影像數量。

這一步讓讀者知道實驗使用了哪些類別、每一類有多少樣本，以及 ground truth mask 是否已經放入。

In [ ]:
df = build_dataset_index()

if df.empty:
    raise RuntimeError("沒有找到任何影像。請確認 data/good 與 data/defect/* 中已放入圖像。")

display(df.head())

summary = df.groupby(["label", "defect_type"]).size().reset_index(name="count")
display(summary)

gt_records = []
for defect_type in DEFECT_TYPES:
    gt_folder = GT_DIR / defect_type
    count = len(list_images(gt_folder)) if gt_folder.exists() else 0
    gt_records.append({"defect_type": defect_type, "ground_truth_count": count})

gt_summary = pd.DataFrame(gt_records)
display(gt_summary)

## 5. Visual Inspection：正常樣本與瑕疵樣本觀察

第一版方法的假設來自對圖像的直接觀察。

正常樣本中的小黑點有幾個特徵：

- 數量多
- 尺寸小
- 分布在整張磁磚表面
- 多數不形成單一大面積連續區域

瑕疵樣本則通常有更明顯的局部異常，例如：

- `crack`：深色、長條、連續
- `glue_strip`：半透明、大面積、局部材質感不同
- `gray_stroke`：局部灰暗、區域集中
- `oil`：偏黃、半透明、顏色與背景不同
- `rough`：局部紋理與反光特徵不同

In [ ]:
N_SHOW_PER_TYPE = 3
items = []

good_samples = df[df["defect_type"] == "good"].head(N_SHOW_PER_TYPE)
for _, row in good_samples.iterrows():
    rgb = read_rgb(row["image_path"])
    items.append((f"good / {row['image_name']}", rgb, None))

for defect_type in DEFECT_TYPES:
    samples = df[df["defect_type"] == defect_type].head(N_SHOW_PER_TYPE)
    for _, row in samples.iterrows():
        rgb = read_rgb(row["image_path"])
        items.append((f"{defect_type} / {row['image_name']}", rgb, None))

show_image_grid(
    items,
    cols=3,
    figsize=(12, max(6, len(items) * 1.7)),
    save_path=FIGURE_DIR / "01_dataset_visual_inspection.png"
)

### Observation

如果只把所有深色區域都當成瑕疵，正常樣本會產生大量 false positives。  
因此這一版不能只是做 `dark pixel detection`，而是要進一步利用 morphology 與 connected components 過濾小型背景紋理。

## 6. Grayscale and Intensity Distribution

因為第一版主要從二值化出發，所以先檢查灰階影像與灰階直方圖。

這一步用來確認：

1. 瑕疵在灰階影像中是否可見  
2. 正常紋理與瑕疵是否有亮度差異  
3. 單純 threshold 是否可能把正常小黑點一起抓進來

In [ ]:
def plot_gray_and_histogram(sample_rows, save_path=None):
    n = len(sample_rows)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.2 * n))

    if n == 1:
        axes = axes.reshape(1, -1)

    for i, (_, row) in enumerate(sample_rows.iterrows()):
        rgb = read_rgb(row["image_path"])
        gray = to_gray(rgb)

        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title(f"Original\n{row['defect_type']} / {row['image_name']}", fontsize=9)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(gray, cmap="gray")
        axes[i, 1].set_title("Grayscale", fontsize=9)
        axes[i, 1].axis("off")

        axes[i, 2].hist(gray.ravel(), bins=50)
        axes[i, 2].set_title("Grayscale Histogram", fontsize=9)
        axes[i, 2].set_xlabel("Intensity")
        axes[i, 2].set_ylabel("Pixel count")

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


selected = []
if not df[df["defect_type"] == "good"].empty:
    selected.append(df[df["defect_type"] == "good"].iloc[0])

for defect_type in DEFECT_TYPES:
    subset = df[df["defect_type"] == defect_type]
    if not subset.empty:
        selected.append(subset.iloc[0])

selected_df = pd.DataFrame(selected)
plot_gray_and_histogram(selected_df, save_path=FIGURE_DIR / "02_grayscale_histogram_comparison.png")

### Intensity Analysis

灰階圖與直方圖通常會顯示一個現象：正常 tile 本身已經有大量深色紋理，因此暗區不等於瑕疵。  
`crack` 通常在灰階中非常明顯；`oil`、`glue_strip`、`rough` 則可能需要顏色、局部對比或紋理特徵才能穩定處理。

## 7. Simple Thresholding Comparison

接著比較三種二值化方式：

1. **Manual dark threshold**：直接設定灰階閾值，例如 `gray < 80`。  
2. **Otsu threshold**：自動根據灰階分布找 threshold。  
3. **Adaptive threshold**：根據局部區域做二值化。

這一節只觀察 raw binary mask，還不加入 morphology。

In [ ]:
def compare_threshold_methods(row, threshold_value=80, adaptive_block_size=51, adaptive_c=5, save_path=None):
    rgb = read_rgb(row["image_path"])
    gray = to_gray(rgb)

    masks = {
        f"Manual dark\nT={threshold_value}": manual_dark_threshold(gray, threshold_value=threshold_value),
        "Otsu dark": otsu_dark_threshold(gray),
        f"Adaptive dark\nblock={adaptive_block_size}, C={adaptive_c}": adaptive_dark_threshold(
            gray, block_size=adaptive_block_size, c=adaptive_c
        )
    }

    fig, axes = plt.subplots(2, 4, figsize=(15, 7))

    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title(f"Original\n{row['defect_type']} / {row['image_name']}", fontsize=9)
    axes[0, 0].axis("off")

    axes[1, 0].imshow(gray, cmap="gray")
    axes[1, 0].set_title("Grayscale", fontsize=9)
    axes[1, 0].axis("off")

    for idx, (title, mask) in enumerate(masks.items(), start=1):
        axes[0, idx].imshow(mask, cmap="gray")
        axes[0, idx].set_title(title, fontsize=9)
        axes[0, idx].axis("off")

        overlay = overlay_mask(rgb, mask, color=(255, 0, 0), alpha=0.45)
        axes[1, idx].imshow(overlay)
        axes[1, idx].set_title("Overlay", fontsize=9)
        axes[1, idx].axis("off")

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


for defect_type in ["good"] + DEFECT_TYPES:
    subset = df[df["defect_type"] == defect_type]
    if subset.empty:
        continue
    row = subset.iloc[0]
    compare_threshold_methods(
        row,
        threshold_value=80,
        adaptive_block_size=51,
        adaptive_c=5,
        save_path=FIGURE_DIR / f"03_threshold_methods_{defect_type}.png"
    )

### Thresholding Observation

Manual threshold 可解釋但高度依賴參數。  
Otsu 不一定適合這個問題，因為 tile 的正常紋理本身就有很多深色斑點。  
Adaptive threshold 對局部變化敏感，但也容易把 good sample 的正常紋理抓成 false positives。

因此，二值化只是候選區域產生的第一步，後續必須加入 morphology 與 connected components filtering。

## 8. Morphology：移除正常小斑點與連接瑕疵區域

這一節是本版方法的核心。

- **Opening**：移除小型孤立區域，減少正常小黑點造成的 false positives。  
- **Closing**：連接破碎的 defect mask，補上裂痕或大面積瑕疵內部的小洞。

這裡會比較 raw mask、opening、closing、opening + closing 的差異。

In [ ]:
def compare_morphology(row, threshold_value=80, open_kernel=5, close_kernel=15, save_path=None):
    rgb = read_rgb(row["image_path"])
    gray = to_gray(rgb)
    raw = manual_dark_threshold(gray, threshold_value=threshold_value)

    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel, open_kernel))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))

    raw_u8 = raw.astype(np.uint8) * 255
    opening = cv2.morphologyEx(raw_u8, cv2.MORPH_OPEN, k_open, iterations=1)
    closing = cv2.morphologyEx(raw_u8, cv2.MORPH_CLOSE, k_close, iterations=1)
    open_close = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, k_close, iterations=1)

    results = [
        ("Original", rgb, None),
        (f"Raw threshold\nT={threshold_value}", raw, "gray"),
        (f"Opening\nkernel={open_kernel}", opening > 0, "gray"),
        (f"Closing\nkernel={close_kernel}", closing > 0, "gray"),
        ("Opening + Closing", open_close > 0, "gray"),
        ("Overlay final", overlay_mask(rgb, (open_close > 0).astype(np.uint8)), None),
    ]

    show_image_grid(results, cols=3, figsize=(12, 7), save_path=save_path)


for defect_type in ["good"] + DEFECT_TYPES:
    subset = df[df["defect_type"] == defect_type]
    if subset.empty:
        continue

    row = subset.iloc[0]
    compare_morphology(
        row,
        threshold_value=80,
        open_kernel=5,
        close_kernel=15,
        save_path=FIGURE_DIR / f"04_morphology_{defect_type}.png"
    )

### Morphology Observation

Morphology 在這裡不是單純讓 mask 變漂亮，而是在實作一個假設：

> 正常紋理通常是小型、零碎、分散的區域；瑕疵則更可能是較大、連續、集中或結構明顯的區域。

但它也有風險：kernel 太大可能破壞細裂痕；closing 太強可能把正常斑點連成大塊。

## 9. Connected Components：從 Mask 到可定位瑕疵候選區域

二值 mask 只是 pixel-level 候選結果。  
這裡將 morphology 後的 mask 轉成可解釋的區域資訊：

- 候選區域數量
- 每個候選區域面積
- bounding box
- 最大候選區域
- 候選區域總面積比例

In [ ]:
DEFAULT_PARAMS = {
    "threshold_method": "manual",
    "threshold_value": 80,
    "open_kernel": 5,
    "close_kernel": 15,
    "min_area": 500
}

def visualize_pipeline_for_row(row, params=DEFAULT_PARAMS, save_path=None):
    rgb = read_rgb(row["image_path"])
    result = run_threshold_morphology_pipeline(rgb, **params)

    gt_path = find_ground_truth(row["image_path"], row["defect_type"])
    gt_mask = read_gt_mask(gt_path, result["gray"].shape) if gt_path else None

    items = [
        ("Original", rgb, None),
        ("Grayscale", result["gray"], "gray"),
        ("Raw threshold mask", result["raw_mask"], "gray"),
        ("Morphology mask", result["morph_mask"], "gray"),
        ("Final filtered mask", result["final_mask"], "gray"),
        ("Final overlay + boxes", result["boxed"], None),
    ]

    if gt_mask is not None:
        pred_overlay = overlay_mask(rgb, result["final_mask"], color=(255, 0, 0), alpha=0.45)
        gt_overlay = overlay_mask(rgb, gt_mask, color=(0, 255, 0), alpha=0.45)
        items.extend([
            ("Ground truth mask", gt_mask, "gray"),
            ("GT overlay", gt_overlay, None),
            ("Prediction overlay", pred_overlay, None),
        ])

    show_image_grid(items, cols=3, figsize=(13, 10), save_path=save_path)

    return result, gt_mask


for defect_type in ["good"] + DEFECT_TYPES:
    subset = df[df["defect_type"] == defect_type]
    if subset.empty:
        continue

    row = subset.iloc[0]
    result, gt_mask = visualize_pipeline_for_row(
        row,
        params=DEFAULT_PARAMS,
        save_path=FIGURE_DIR / f"05_full_pipeline_{defect_type}.png"
    )

### Pipeline Observation

完整流程圖應該用來觀察三件事：

1. raw threshold 是否已經包含 defect。  
2. morphology 是否移除了正常紋理。  
3. final mask 是否和 ground truth 有合理重疊。

比對時不能只看 prediction mask 和 ground truth mask，也要回到原圖看瑕疵本身的外觀。

## 10. Prediction vs Ground Truth：三層式比對方式

本 Notebook 使用三層式比對：

1. **Original image**：先確認瑕疵在真實影像中的外觀。  
2. **Prediction / Ground Truth Overlay**：將 prediction 和 ground truth 疊回原圖，觀察位置是否對應。  
3. **Error Map**：用不同顏色顯示 TP、FP、FN。

Error map 顏色定義：

- 綠色：TP，prediction 與 ground truth 重疊  
- 紅色：FP，prediction 有抓到但 ground truth 沒標  
- 藍色：FN，ground truth 有標但 prediction 沒抓到

In [ ]:
def make_error_map(pred_mask, gt_mask):
    """
    建立 TP / FP / FN error map。
    TP: green, FP: red, FN: blue
    """
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    error = np.zeros((*pred.shape, 3), dtype=np.uint8)
    tp = np.logical_and(pred, gt)
    fp = np.logical_and(pred, ~gt)
    fn = np.logical_and(~pred, gt)

    error[tp] = (0, 255, 0)
    error[fp] = (255, 0, 0)
    error[fn] = (0, 120, 255)

    return error


def compare_prediction_with_gt(row, params=DEFAULT_PARAMS, save_path=None):
    rgb = read_rgb(row["image_path"])
    result = run_threshold_morphology_pipeline(rgb, **params)

    gt_path = find_ground_truth(row["image_path"], row["defect_type"])
    gt_mask = read_gt_mask(gt_path, result["gray"].shape) if gt_path else None

    if gt_mask is None:
        print(f"找不到 ground truth：{row['defect_type']} / {row['image_name']}")
        return None

    pred_mask = result["final_mask"]
    metrics = compute_mask_metrics(pred_mask, gt_mask)
    error_map = make_error_map(pred_mask, gt_mask)

    pred_overlay = overlay_mask(rgb, pred_mask, color=(255, 0, 0), alpha=0.45)
    gt_overlay = overlay_mask(rgb, gt_mask, color=(0, 255, 0), alpha=0.45)

    error_overlay = rgb.copy()
    error_pixels = error_map.sum(axis=2) > 0
    blended = cv2.addWeighted(rgb, 0.45, error_map, 0.55, 0)
    error_overlay[error_pixels] = blended[error_pixels]

    items = [
        ("Original", rgb, None),
        ("Prediction mask", pred_mask, "gray"),
        ("Ground truth mask", gt_mask, "gray"),
        ("Prediction overlay", pred_overlay, None),
        ("Ground truth overlay", gt_overlay, None),
        ("Error map\nTP=green, FP=red, FN=blue", error_overlay, None),
    ]

    show_image_grid(items, cols=3, figsize=(13, 8), save_path=save_path)

    print("Metrics:")
    for k in ["iou", "dice", "precision", "recall", "false_positive_rate"]:
        print(f"{k}: {metrics[k]:.4f}")

    return metrics


metrics_records = []

for defect_type in DEFECT_TYPES:
    subset = df[df["defect_type"] == defect_type].head(3)
    for _, row in subset.iterrows():
        metrics = compare_prediction_with_gt(
            row,
            params=DEFAULT_PARAMS,
            save_path=FIGURE_DIR / f"06_pred_vs_gt_{defect_type}_{row['image_path'].stem}.png"
        )
        if metrics is not None:
            record = {
                "image_name": row["image_name"],
                "defect_type": defect_type,
                **metrics
            }
            metrics_records.append(record)

metrics_df = pd.DataFrame(metrics_records)
if not metrics_df.empty:
    display(metrics_df)

### Ground Truth Comparison Analysis

若 IoU 或 Dice 不高，不一定代表方法完全沒用。  
對這一版而言，比較重要的是判斷：

1. 是否抓到正確的大致位置。  
2. FP 是否來自正常小斑點。  
3. FN 是否來自 defect 外觀不符合暗區假設。  
4. prediction 是偏大、偏小、漏抓，還是抓錯正常紋理。

這也是為什麼本 Notebook 不只放 mask，而是同時放 original、overlay 與 error map。

## 11. Good Sample False Positive Check

做 defect detection 不能只展示瑕疵圖像。  
如果方法在 good sample 上也產生大量候選區域，那它就不適合作為穩定檢測流程。

這一節使用 good images 測試同一套流程。

In [ ]:
good_records = []

good_subset = df[df["defect_type"] == "good"].head(6)

for _, row in good_subset.iterrows():
    rgb = read_rgb(row["image_path"])
    result = run_threshold_morphology_pipeline(rgb, **DEFAULT_PARAMS)

    components = result["components"]
    total_area = int(result["final_mask"].sum())
    image_area = result["final_mask"].shape[0] * result["final_mask"].shape[1]
    area_ratio = total_area / image_area

    good_records.append({
        "image_name": row["image_name"],
        "num_candidates": len(components),
        "total_candidate_area": total_area,
        "candidate_area_ratio": area_ratio,
        "largest_component_area": max([c["area"] for c in components], default=0)
    })

    items = [
        ("Original", rgb, None),
        ("Raw threshold", result["raw_mask"], "gray"),
        ("Morphology mask", result["morph_mask"], "gray"),
        ("Final mask", result["final_mask"], "gray"),
        ("Overlay + boxes", result["boxed"], None),
    ]
    show_image_grid(
        items,
        cols=5,
        figsize=(15, 3.3),
        save_path=FIGURE_DIR / f"07_good_false_positive_{row['image_path'].stem}.png"
    )

good_result_df = pd.DataFrame(good_records)
display(good_result_df)

### Good Sample Analysis

理想情況是 raw threshold 抓到許多正常深色斑點，但 morphology 與 area filtering 後大部分被移除。  
若 good sample 仍留下大型候選區域，可能代表 threshold 太高、closing 太強、min area 太低，或正常 tile 紋理本身變化太大。

## 12. Apply to All Defect Types

接著將同一組參數套用到每一種瑕疵上，觀察這一版方法對不同 defect type 的適用性。

這裡不針對每類手動調參，因為第一版想測的是：  
一套簡單的 threshold + morphology 流程，是否能對多種 tile defects 產生合理候選區域？

In [ ]:
all_records = []
MAX_PER_TYPE = 5

for defect_type in DEFECT_TYPES:
    subset = df[df["defect_type"] == defect_type].head(MAX_PER_TYPE)

    for _, row in subset.iterrows():
        rgb = read_rgb(row["image_path"])
        result = run_threshold_morphology_pipeline(rgb, **DEFAULT_PARAMS)

        gt_path = find_ground_truth(row["image_path"], row["defect_type"])
        gt_mask = read_gt_mask(gt_path, result["gray"].shape) if gt_path else None
        metrics = compute_mask_metrics(result["final_mask"], gt_mask) if gt_mask is not None else {}

        total_area = int(result["final_mask"].sum())
        image_area = result["final_mask"].shape[0] * result["final_mask"].shape[1]
        area_ratio = total_area / image_area

        record = {
            "image_name": row["image_name"],
            "defect_type": defect_type,
            "num_candidates": len(result["components"]),
            "total_candidate_area": total_area,
            "candidate_area_ratio": area_ratio,
            "largest_component_area": max([c["area"] for c in result["components"]], default=0),
            "has_ground_truth": gt_mask is not None
        }

        if metrics:
            record.update({
                "iou": metrics["iou"],
                "dice": metrics["dice"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "false_positive_rate": metrics["false_positive_rate"]
            })

        all_records.append(record)

        items = [
            ("Original", rgb, None),
            ("Raw threshold", result["raw_mask"], "gray"),
            ("Final mask", result["final_mask"], "gray"),
            ("Overlay + boxes", result["boxed"], None),
        ]

        if gt_mask is not None:
            error_overlay = rgb.copy()
            error_map = make_error_map(result["final_mask"], gt_mask)
            error_pixels = error_map.sum(axis=2) > 0
            blended = cv2.addWeighted(rgb, 0.45, error_map, 0.55, 0)
            error_overlay[error_pixels] = blended[error_pixels]
            items.append(("Error map\nTP green / FP red / FN blue", error_overlay, None))

        show_image_grid(
            items,
            cols=len(items),
            figsize=(4 * len(items), 4),
            save_path=FIGURE_DIR / f"08_all_types_{defect_type}_{row['image_path'].stem}.png"
        )

all_result_df = pd.DataFrame(all_records)
display(all_result_df)

In [ ]:
if not all_result_df.empty:
    numeric_cols = [
        "num_candidates",
        "total_candidate_area",
        "candidate_area_ratio",
        "largest_component_area",
        "iou",
        "dice",
        "precision",
        "recall",
        "false_positive_rate"
    ]
    existing_numeric_cols = [c for c in numeric_cols if c in all_result_df.columns]

    summary_by_type = all_result_df.groupby("defect_type")[existing_numeric_cols].mean(numeric_only=True).reset_index()
    display(summary_by_type)

    all_result_df.to_csv(TABLE_DIR / "threshold_morphology_results.csv", index=False, encoding="utf-8-sig")
    summary_by_type.to_csv(TABLE_DIR / "threshold_morphology_summary_by_type.csv", index=False, encoding="utf-8-sig")

    print("Saved:")
    print(TABLE_DIR / "threshold_morphology_results.csv")
    print(TABLE_DIR / "threshold_morphology_summary_by_type.csv")

## 13. Parameter Sensitivity Analysis

這一版方法高度依賴參數，因此必須測試參數敏感度。

主要觀察三個參數：

1. `threshold_value`：控制哪些暗區會進入 raw mask。  
2. `open_kernel`：控制小紋理是否被移除。  
3. `min_area`：控制多小的區域會被視為正常紋理並被移除。

這一節的目的不是找到唯一最佳參數，而是看出方法的 trade-off。

In [ ]:
def parameter_sensitivity_threshold(row, threshold_values=[50, 65, 80, 95, 110], save_path=None):
    rgb = read_rgb(row["image_path"])
    items = [("Original", rgb, None)]

    for t in threshold_values:
        result = run_threshold_morphology_pipeline(
            rgb,
            threshold_method="manual",
            threshold_value=t,
            open_kernel=DEFAULT_PARAMS["open_kernel"],
            close_kernel=DEFAULT_PARAMS["close_kernel"],
            min_area=DEFAULT_PARAMS["min_area"]
        )
        items.append((f"T={t}\nfinal mask", result["final_mask"], "gray"))
        items.append((f"T={t}\noverlay", result["boxed"], None))

    show_image_grid(
        items,
        cols=3,
        figsize=(13, 3.5 * math.ceil(len(items) / 3)),
        save_path=save_path
    )


for defect_type in ["crack", "gray_stroke", "oil", "rough"]:
    subset = df[df["defect_type"] == defect_type]
    if subset.empty:
        continue

    row = subset.iloc[0]
    parameter_sensitivity_threshold(
        row,
        threshold_values=[50, 65, 80, 95, 110],
        save_path=FIGURE_DIR / f"09_threshold_sensitivity_{defect_type}.png"
    )

In [ ]:
def parameter_sensitivity_morphology(row, open_kernels=[3, 5, 7, 11], close_kernel=15, save_path=None):
    rgb = read_rgb(row["image_path"])
    items = [("Original", rgb, None)]

    for k in open_kernels:
        result = run_threshold_morphology_pipeline(
            rgb,
            threshold_method="manual",
            threshold_value=DEFAULT_PARAMS["threshold_value"],
            open_kernel=k,
            close_kernel=close_kernel,
            min_area=DEFAULT_PARAMS["min_area"]
        )
        items.append((f"open={k}\nfinal mask", result["final_mask"], "gray"))
        items.append((f"open={k}\noverlay", result["boxed"], None))

    show_image_grid(
        items,
        cols=3,
        figsize=(13, 3.5 * math.ceil(len(items) / 3)),
        save_path=save_path
    )


for defect_type in ["good", "crack", "gray_stroke"]:
    subset = df[df["defect_type"] == defect_type]
    if subset.empty:
        continue

    row = subset.iloc[0]
    parameter_sensitivity_morphology(
        row,
        open_kernels=[3, 5, 7, 11],
        close_kernel=15,
        save_path=FIGURE_DIR / f"10_morphology_sensitivity_{defect_type}.png"
    )

In [ ]:
def parameter_sensitivity_min_area(row, min_areas=[100, 300, 500, 1000, 2000], save_path=None):
    rgb = read_rgb(row["image_path"])
    items = [("Original", rgb, None)]

    for area in min_areas:
        result = run_threshold_morphology_pipeline(
            rgb,
            threshold_method="manual",
            threshold_value=DEFAULT_PARAMS["threshold_value"],
            open_kernel=DEFAULT_PARAMS["open_kernel"],
            close_kernel=DEFAULT_PARAMS["close_kernel"],
            min_area=area
        )
        items.append((f"min_area={area}\nfinal mask", result["final_mask"], "gray"))
        items.append((f"min_area={area}\noverlay", result["boxed"], None))

    show_image_grid(
        items,
        cols=3,
        figsize=(13, 3.5 * math.ceil(len(items) / 3)),
        save_path=save_path
    )


for defect_type in ["good", "crack", "glue_strip", "gray_stroke"]:
    subset = df[df["defect_type"] == defect_type]
    if subset.empty:
        continue

    row = subset.iloc[0]
    parameter_sensitivity_min_area(
        row,
        min_areas=[100, 300, 500, 1000, 2000],
        save_path=FIGURE_DIR / f"11_min_area_sensitivity_{defect_type}.png"
    )

### Parameter Sensitivity Analysis

參數敏感度反映這一版方法的核心 trade-off：

- threshold 太低：只保留非常暗的區域，可能漏掉 gray stroke、oil、rough。  
- threshold 太高：正常小黑點大量進入 mask，false positives 增加。  
- opening kernel 太小：無法有效移除正常紋理。  
- opening kernel 太大：可能破壞細裂痕或小型 defect。  
- min area 太低：保留太多正常小區域。  
- min area 太高：可能漏掉小型或細長 defect。

對 crack 來說，單純 area 不一定是最好的條件，因為裂痕可能面積不大但長度很長。下一版可以加入 aspect ratio、skeleton length 或 edge-based features。

## 14. Result Discussion by Defect Type

### Crack

`crack` 通常是這一版最容易處理的類型之一。  
原因是裂痕具有深色、細長、連續的特徵，和正常小斑點相比有明顯結構差異。

但若 opening kernel 太大，細裂痕可能會被削弱。  
因此 crack 類型下一版可以加入 edge detection 或 line structure filtering，而不是只靠 area。

---

### Glue Strip

`glue_strip` 的問題在於它不一定是深色區域，而是半透明或亮灰色區域。  
如果二值化只針對暗區，它可能只能抓到部分邊界或局部陰影。

下一版應該加入 bright anomaly mask、background subtraction 或 local contrast map。

---

### Gray Stroke

`gray_stroke` 通常具有較大面積，因此 morphology 和 area filtering 可能有效。  
但如果灰色區域和正常背景亮度接近，單一 threshold 可能不穩。

下一版可以用 large-kernel Gaussian blur 建立背景，再使用 original - background 的 difference map。

---

### Oil

`oil` 的可見特徵常常不只是亮度，而是顏色偏移與半透明材質。  
灰階 threshold 可能會低估油漬區域。

下一版應該加入 HSV 或 Lab 色彩空間分析。

---

### Rough

`rough` 比較像局部 texture response 或反光變化，不一定能靠暗區 threshold 穩定偵測。  
這類缺陷更適合使用 local variance、Laplacian response 或 high-frequency response。

---

### Good

good sample 的表現決定這一版方法是否有實用性。  
若 good sample 經過 morphology 後仍產生大量 candidate，代表方法無法有效區分背景紋理與 defect。

## 15. Conclusion

這一版實驗並沒有能夠驗證了最初的直覺想法：

> 正常 tile 的深色小斑點可以透過 morphology 與 connected components filtering 移除一部分，而較大、較連續、結構較明顯的瑕疵有機會被保留下來。

### 本版方法的限制

- 對 threshold value 敏感。
- 對顏色型瑕疵不夠穩定。
- 對低對比、半透明或 texture-based defects 效果有限。
- 單純 area filtering 不適合所有瑕疵，例如細長裂痕。
- morphology kernel 過大可能破壞小瑕疵，過小則無法移除正常紋理。

### 下一版改善方向

從結果圖中可以清楚觀察出這個作法太過簡略，基本上沒有任何一種瑕疵是可以被正確檢測出來的，這也更說明了，應該是要針對不同瑕疵做不同針對性的處理才是比較合適的做法。

## 16. Optional：儲存預測 Mask 與 Overlay

若要將這一版結果輸出成檔案，可以執行下面這段程式。  
它會將每張影像的 final mask 與 overlay 儲存到 `outputs/masks/` 與 `outputs/overlays/`。

In [ ]:
SAVE_ALL_RESULTS = True

if SAVE_ALL_RESULTS:
    for _, row in df.iterrows():
        rgb = read_rgb(row["image_path"])
        result = run_threshold_morphology_pipeline(rgb, **DEFAULT_PARAMS)

        stem = row["image_path"].stem
        defect_type = row["defect_type"]

        mask_out_dir = MASK_DIR / defect_type
        overlay_out_dir = OVERLAY_DIR / defect_type
        mask_out_dir.mkdir(parents=True, exist_ok=True)
        overlay_out_dir.mkdir(parents=True, exist_ok=True)

        mask_path = mask_out_dir / f"{stem}_pred_mask.png"
        overlay_path = overlay_out_dir / f"{stem}_overlay.png"

        cv2.imwrite(str(mask_path), result["final_mask"].astype(np.uint8) * 255)
        cv2.imwrite(str(overlay_path), cv2.cvtColor(result["boxed"], cv2.COLOR_RGB2BGR))

    print("All prediction masks and overlays have been saved.")
else:
    print("SAVE_ALL_RESULTS is False. Set it to True if you want to export all masks and overlays.")